# QDSV/QIntent Random Sparse qLDPC-Style Benchmark

This notebook moves beyond the controlled benchmark. It generates a random sparse syndrome map, samples random error patterns, enumerates low-weight syndrome-compatible correction candidates, compares a minimum-weight / likelihood baseline with QDSV structured semantic reranking, and reports aggregate tradeoffs.

Unlike the controlled benchmark, this experiment is not designed for QDSV to win every case. The goal is to measure when QDSV improves risk, ties, or worsens outcomes, and what tradeoff appears between risk reduction, exact correction, and a logical-failure proxy.

This is still a toy sparse-check benchmark, not a production qLDPC decoder benchmark.

## 1. Install QIntent

In [ ]:
!pip install -q --upgrade "git+https://github.com/qdsvquantum-afk/qintent.git"

In [ ]:
import json
import math
import random
from itertools import combinations
from pathlib import Path

import pandas as pd

from qintent import QIntentClient

API_URL = "https://qdsv-api-568788785178.us-central1.run.app/api"
client = QIntentClient(api_url=API_URL, timeout=60)

client.spec()["public_limits"]

## 2. Random sparse benchmark configuration

`N_SAMPLES` controls public API usage. Each sample performs one QIntent `run()` call. Keep it moderate for public-preview quota.

In [ ]:
SEED = 260716
N_QUBITS = 18
M_CHECKS = 8
N_SAMPLES = 20
MAX_CANDIDATE_WEIGHT = 3
TRUE_ERROR_MAX_WEIGHT = 2
COLUMN_WEIGHT = 3
PHYSICAL_ERROR_RATE = 0.08

rng = random.Random(SEED)

## 3. Generate random sparse check columns

In [ ]:
check_columns = {}
used_columns = set()

for q in range(N_QUBITS):
    while True:
        weight = COLUMN_WEIGHT if rng.random() < 0.65 else 2
        positions = rng.sample(range(M_CHECKS), weight)
        column = [0] * M_CHECKS
        for pos in positions:
            column[pos] = 1
        column = tuple(column)
        if column not in used_columns and any(column):
            used_columns.add(column)
            check_columns[q] = column
            break

logical_sensitive_qubits = set(rng.sample(range(N_QUBITS), max(3, N_QUBITS // 4)))

pd.DataFrame([
    {
        "qubit": q,
        "check_column": "".join(str(bit) for bit in column),
        "logical_sensitive": q in logical_sensitive_qubits,
    }
    for q, column in check_columns.items()
])

## 4. Candidate generation and labels

The benchmark generates random true errors of weight 1 or 2, computes their syndrome, and enumerates low-weight correction candidates with the same syndrome.

Labels such as `exact_correction` and `logical_failure_proxy` are used for evaluation only. They are included in rows for traceability, but they are not used as public QIntent signals.

In [ ]:
def xor_syndrome(qubits):
    out = [0] * M_CHECKS
    for q in qubits:
        out = [a ^ b for a, b in zip(out, check_columns[q])]
    return tuple(out)


def residual_error(true_error, correction):
    return frozenset(set(true_error) ^ set(correction))


def logical_failure_proxy(residual):
    # Toy proxy: a small residual touching a logical-sensitive qubit is treated as risky.
    return bool(residual & logical_sensitive_qubits) and len(residual) <= 3


def generate_candidates(scenario_id, true_error):
    observed = xor_syndrome(true_error)
    raw = []

    for weight in range(1, MAX_CANDIDATE_WEIGHT + 1):
        for qubits in combinations(range(N_QUBITS), weight):
            if xor_syndrome(qubits) != observed:
                continue

            overlap = sum(q in logical_sensitive_qubits for q in qubits)
            log_likelihood = weight * math.log(PHYSICAL_ERROR_RATE) + (N_QUBITS - weight) * math.log(1 - PHYSICAL_ERROR_RATE)
            residual = residual_error(true_error, qubits)
            raw.append((qubits, weight, overlap, log_likelihood, len(residual) == 0, logical_failure_proxy(residual)))

    if len(raw) < 2:
        return []

    likelihoods = [item[3] for item in raw]
    low, high = min(likelihoods), max(likelihoods)

    rows = []
    for candidate_index, (qubits, weight, overlap, log_likelihood, exact, logical_fail) in enumerate(raw):
        decoder_confidence = 1000 if high == low else int(round(600 + 400 * (log_likelihood - low) / (high - low)))
        rows.append({
            "scenario_id": scenario_id,
            "candidate_index": candidate_index,
            "correction_qubits": " ".join(str(q) for q in qubits),
            "candidate_weight": weight,
            "observed_syndrome": "".join(str(bit) for bit in observed),
            "predicted_syndrome": "".join(str(bit) for bit in observed),
            "syndrome_support": max(0, 1000 - 25 * max(0, weight - 1)),
            "check_consistency": 1000,
            "logical_preservation": max(0, 950 - 180 * overlap - 20 * max(0, weight - 1)),
            "distance_safety": max(0, 930 - 160 * overlap - 15 * weight),
            "decoder_confidence": decoder_confidence,
            "propagation_safety": max(0, 930 - 50 * weight - 50 * overlap),
            "syndrome_risk": min(1000, 30 + 12 * weight),
            "logical_risk": min(1000, 25 + 170 * overlap + 18 * weight),
            "syndrome_entropy_adjustment": 15 if weight == 2 and overlap == 0 else (-15 if overlap else 0),
            "exact_correction": exact,
            "logical_failure_proxy": logical_fail,
        })
    return rows

## 5. QIntent structured semantic score

In [ ]:
source = """
find_rows("candidate_index")
  .using_structured_semantic_score([
      block("syndrome", [
          signal("syndrome_support", influence=30, priority=2),
          signal("check_consistency", influence=20, priority=1),
      ], influence=30, priority=2, risk_adjustment="syndrome_risk", adjustments=[
          adjustment("syndrome_entropy_adjustment", influence=5),
      ]),
      block("logical_safety", [
          signal("logical_preservation", influence=40, priority=3),
          signal("distance_safety", influence=20, priority=2),
      ], influence=40, priority=3, risk_adjustment="logical_risk"),
      block("decoder", [
          signal("decoder_confidence", influence=25, priority=1),
          signal("propagation_safety", influence=15, priority=2),
      ], influence=30, priority=1),
  ], global_risk="logical_risk", profile="qldpc_random_sparse_benchmark")
  .accept_if(threshold=600)
  .rank()
  .top_k(1)
"""

print(source)

## 6. Run random benchmark

This benchmark is intentionally less controlled. It should show improvements, ties, and possibly worse cases.

In [ ]:
summary_rows = []
selected_by_scenario = []
scenario_rows = []
attempts = 0
scenario_id = 0

while scenario_id < N_SAMPLES and attempts < 1000:
    attempts += 1
    true_weight = 1 if rng.random() < 0.65 else 2
    true_error = tuple(sorted(rng.sample(range(N_QUBITS), true_weight)))
    rows = generate_candidates(scenario_id, true_error)
    if len(rows) < 2:
        continue

    candidates = pd.DataFrame(rows)
    baseline = candidates.sort_values(["decoder_confidence", "candidate_weight"], ascending=[False, True]).iloc[0]
    result = client.run(source, rows=rows)
    selected = result["result"]["selected_rows"][0]

    risk_delta = int(baseline["logical_risk"]) - int(selected["logical_risk"])
    exact_delta = int(bool(selected["exact_correction"])) - int(bool(baseline["exact_correction"]))
    failure_delta = int(bool(baseline["logical_failure_proxy"])) - int(bool(selected["logical_failure_proxy"]))
    outcome = "improved_risk" if risk_delta > 0 else ("worse_risk" if risk_delta < 0 else "tied_risk")

    summary_rows.append({
        "scenario_id": scenario_id,
        "true_error": " ".join(str(q) for q in true_error),
        "true_weight": true_weight,
        "candidate_count": len(rows),
        "baseline_qubits": baseline["correction_qubits"],
        "baseline_confidence": int(baseline["decoder_confidence"]),
        "baseline_logical_risk": int(baseline["logical_risk"]),
        "baseline_exact": bool(baseline["exact_correction"]),
        "baseline_failure": bool(baseline["logical_failure_proxy"]),
        "qdsv_qubits": selected["correction_qubits"],
        "qdsv_confidence": int(selected["decoder_confidence"]),
        "qdsv_logical_risk": int(selected["logical_risk"]),
        "qdsv_exact": bool(selected["exact_correction"]),
        "qdsv_failure": bool(selected["logical_failure_proxy"]),
        "risk_delta": risk_delta,
        "exact_delta": exact_delta,
        "failure_delta": failure_delta,
        "outcome": outcome,
    })
    selected_by_scenario.append(selected)
    scenario_rows.append(rows)
    scenario_id += 1

summary = pd.DataFrame(summary_rows)
summary

## 7. Aggregate metrics and tradeoffs

In [ ]:
outcome_counts = summary.groupby("outcome").size().to_dict()

metrics = {
    "samples": int(len(summary)),
    "attempts": int(attempts),
    "outcome_counts": outcome_counts,
    "baseline_exact_rate": float(summary["baseline_exact"].mean()),
    "qdsv_exact_rate": float(summary["qdsv_exact"].mean()),
    "baseline_failure_proxy_rate": float(summary["baseline_failure"].mean()),
    "qdsv_failure_proxy_rate": float(summary["qdsv_failure"].mean()),
    "baseline_avg_logical_risk": float(summary["baseline_logical_risk"].mean()),
    "qdsv_avg_logical_risk": float(summary["qdsv_logical_risk"].mean()),
    "avg_risk_delta": float(summary["risk_delta"].mean()),
    "avg_exact_delta": float(summary["exact_delta"].mean()),
    "avg_failure_delta": float(summary["failure_delta"].mean()),
}

metrics

## 8. Inspect improved, tied, and worse cases

In [ ]:
summary.sort_values(["risk_delta", "exact_delta", "failure_delta"], ascending=[False, False, False]).head(10)

In [ ]:
summary.sort_values(["risk_delta", "exact_delta", "failure_delta"], ascending=[True, True, True]).head(10)

## 9. Save evidence

In [ ]:
evidence = {
    "api_url": API_URL,
    "profile": "qldpc_random_sparse_benchmark",
    "benchmark_design": {
        "seed": SEED,
        "n_qubits": N_QUBITS,
        "m_checks": M_CHECKS,
        "n_samples": N_SAMPLES,
        "max_candidate_weight": MAX_CANDIDATE_WEIGHT,
        "true_error_max_weight": TRUE_ERROR_MAX_WEIGHT,
        "physical_error_rate": PHYSICAL_ERROR_RATE,
        "column_weight": COLUMN_WEIGHT,
        "construction": "random sparse check columns with random error samples and low-weight syndrome-compatible candidate enumeration",
    },
    "check_columns": {str(k): list(v) for k, v in check_columns.items()},
    "logical_sensitive_qubits": sorted(logical_sensitive_qubits),
    "qintent_source": source,
    "summary": summary.to_dict(orient="records"),
    "metrics": metrics,
    "internal_formula_exposed": False,
}

Path("qdsv_qldpc_random_sparse_benchmark_evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
summary.to_csv("qdsv_qldpc_random_sparse_benchmark_summary.csv", index=False)

print("Saved:")
print("- qdsv_qldpc_random_sparse_benchmark_evidence.json")
print("- qdsv_qldpc_random_sparse_benchmark_summary.csv")

## Interpretation

This random sparse benchmark is meant to reveal tradeoffs. A useful QDSV post-decoding layer should not be evaluated only on cases where it wins. The key questions are:

- When does QDSV reduce logical-risk score?
- When does it tie the baseline?
- When does risk reduction come at the cost of exact correction?
- When does it reduce or increase the logical-failure proxy?

Those tradeoffs guide the next step: connecting a real decoder and tuning the post-decoding policy against real qLDPC metrics.